
# Gyroid as an implicit Phase

End-to-end demonstration of the implicit-first pipeline of microgen 2.0, with
no CAD detour.

A :class:`microgen.Tpms` exposes a scalar SDF directly; wrapping it in a
:class:`microgen.Phase` via :meth:`Phase.from_shape` gives access to
:meth:`phase.surface_mesh`, :attr:`phase.pieces`, :attr:`phase.center_of_mass`,
and the transforms :meth:`Phase.translate` / :meth:`Phase.scale`,
none of which require the optional ``[cad]`` extra (no OCCT, no STEP, no gmsh).


In [ ]:
import numpy as np
import pyvista as pv

from microgen import Phase, Sphere, Tpms, surface_functions

Build an implicit gyroid TPMS — the field is a callable, no BREP built.



In [ ]:
gyroid = Tpms(
    surface_function=surface_functions.gyroid,
    offset=0.3,
    cell_size=1.0,
    repeat_cell=1,
    resolution=20,
)

Wrap the implicit Shape in a Phase.  Field-backed: no CAD required.



In [ ]:
phase = Phase.from_shape(gyroid, resolution=80)
print(f"Phase bounds: {phase.bounds}")
print(f"Phase period: {phase.period}  (set by Tpms.cell_size)")
print(f"Phase field-backed: {phase.field is not None}")

Marching-cubes surface mesh straight from the SDF.



In [ ]:
surface = phase.surface_mesh()
surface.save("gyroid_phase_surface.stl")
print(f"Surface mesh: {surface.n_cells} cells -> gyroid_phase_surface.stl")

Moments computed by grid quadrature — no OCCT needed.



In [ ]:
print(f"Center of mass:  {phase.center_of_mass}")
print(f"Inertia tensor diagonal:  {np.diag(phase.inertia_matrix)}")

Connected components via Phase.pieces (scipy.ndimage on the SDF grid).



In [ ]:
pieces = phase.pieces
print(f"Pieces: {len(pieces)}  (gyroid 'sheet' part is multiply connected)")
for i, piece in enumerate(pieces[:3]):
    print(f"  piece[{i}]: volume={piece.volume:.4f}, com={piece.com}")

Transforms return a new Phase by default (inplace=False).



In [ ]:
moved = phase.translate((1.0, 0.0, 0.0))
print(f"\nTranslated phase COM: {moved.center_of_mass}  (shifted by +1 in x)")

scaled = phase.scale(2.0)
print(f"Scaled phase bounds:  {scaled.bounds}  (cell doubled)")

Compose with another implicit Shape via F-rep boolean ops.



In [ ]:
hollowed = gyroid - Sphere(radius=0.3)  # SDF difference, returns a new Shape
hollow_phase = Phase.from_shape(hollowed, bounds=phase.bounds, resolution=80)
print(f"\nHollowed phase pieces: {len(hollow_phase.pieces)}")
hollow_phase.surface_mesh().save("gyroid_minus_sphere.stl")
print("Wrote gyroid_minus_sphere.stl")

The gyroid phase and the gyroid minus a sphere, side by side.



In [ ]:
pl = pv.Plotter(shape=(1, 2), window_size=(1200, 600))
pl.add_mesh(surface, smooth_shading=True, color="#ef7d2d")
pl.subplot(0, 1)
pl.add_mesh(hollow_phase.surface_mesh(), smooth_shading=True, color="#ef7d2d")
pl.link_views()
pl.show()